# Azure ML General Notebook
**Purpose:** Reusable starting point for any Azure ML project.

**Prerequisites before running:**
- `pip install azure-ai-ml azure-identity mltable`
- Fill in your values in the **CONFIG** cell below (Section 1)
- Have a `config.json` in this folder OR fill in credentials manually

---

## Section 1 — Connect to Workspace

**Two options:**
- **Option A (recommended):** Place `config.json` next to this notebook (download it from Azure ML Studio → top-right menu → Download config)
- **Option B (manual):** Fill in your subscription/resource group/workspace below

> ⚠️ The original notebook failed here because `config.json` was not found locally. On an Azure ML Compute Instance it is auto-provided; locally you must supply it yourself.

In [13]:
pip install azure-ai-ml azure-identity mltable

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

# ── AUTHENTICATE ─────────────────────────────────────────────────────────────
# Tries environment/CLI/managed identity first; falls back to browser login
try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
    print("✓ DefaultAzureCredential succeeded")
except Exception:
    credential = InteractiveBrowserCredential()
    print("✓ Falling back to InteractiveBrowserCredential")

# Esto no esta en el notebook de AML
# On a Compute Instance, config.json is auto-available — no manual IDs needed
# ml_client = MLClient.from_config(credential=credential)
# print(f"✓ Connected to workspace: {ml_client.workspace_name}")

DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' pa

✓ Falling back to InteractiveBrowserCredential


Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


✓ Connected to workspace: mlw-dp100-l5fb3c26f9c764205ac


In [8]:
# ── OPTION A: config.json (recommended) ──────────────────────────────────────
# Download config.json from Azure ML Studio and place it next to this notebook.
# Then run this cell.

ml_client = MLClient.from_config(credential=credential)

# ── OPTION B: Manual credentials (fallback) ───────────────────────────────────
# Fill in your values if you don't have config.json

# SUBSCRIPTION_ID = "YOUR-SUBSCRIPTION-ID"   # e.g. "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx"
# RESOURCE_GROUP  = "YOUR-RESOURCE-GROUP"     # e.g. "rg-mlworkspace"
# WORKSPACE_NAME  = "YOUR-WORKSPACE-NAME"     # e.g. "my-aml-workspace"

# ml_client = MLClient(
#     credential=credential,
#     subscription_id=SUBSCRIPTION_ID,
#     resource_group_name=RESOURCE_GROUP,
#     workspace_name=WORKSPACE_NAME,
# )

# print(f"✓ Connected to workspace: {ml_client.workspace_name}")

Found the config file in: .\config.json
Overriding of current MeterProvider is not allowed
Overriding of current TracerProvider is not allowed
Overriding of current LoggerProvider is not allowed
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented
Attempting to instrument while already instrumented


---
## Section 2 — Datastores

In [12]:
# List all datastores in the workspace
print("Datastores in workspace:")
for ds in ml_client.datastores.list():
    print(" -", ds.name)



Datastores in workspace:


ClientAuthenticationError: (InvalidAuthenticationTokenTenant) The access token is from the wrong issuer 'https://sts.windows.net/f8cdef31-a31e-4b4a-93e4-5f571e91255a/'. It must match the tenant 'https://sts.windows.net/e43b0362-e932-4a58-9c0d-3071db3f47a8/' associated with this subscription. Please use the authority (URL) 'https://login.windows.net/e43b0362-e932-4a58-9c0d-3071db3f47a8' to get the token. Note, if the subscription is transferred to another tenant there is no impact to the services, but information about new tenant could take time to propagate (up to an hour). If you just transferred your subscription and see this error message, please try back later.
Code: InvalidAuthenticationTokenTenant
Message: The access token is from the wrong issuer 'https://sts.windows.net/f8cdef31-a31e-4b4a-93e4-5f571e91255a/'. It must match the tenant 'https://sts.windows.net/e43b0362-e932-4a58-9c0d-3071db3f47a8/' associated with this subscription. Please use the authority (URL) 'https://login.w

In [ ]:
# Create a new Blob datastore (only run this when you want to register a new storage)
# Fill in YOUR-STORAGE-ACCOUNT-NAME and YOUR-ACCOUNT-KEY before running

from azure.ai.ml.entities import AzureBlobDatastore, AccountKeyConfiguration

store = AzureBlobDatastore(
    name="blob_training_data",
    description="Blob Storage for training data",
    account_name="YOUR-STORAGE-ACCOUNT-NAME",
    container_name="training-data",
    credentials=AccountKeyConfiguration(
        account_key="YOUR-ACCOUNT-KEY"
    ),
)

ml_client.datastores.create_or_update(store)
print(f"✓ Datastore '{store.name}' registered")

---
## Section 3 — Data Assets

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# URI_FILE — points to a single CSV file (uploaded automatically to default datastore)
my_data = Data(
    path="/home/azureuser/cloudfiles/code/Users/Castle2696/azure-ml-labs/data/diabetes.csv",
    type=AssetTypes.URI_FILE,
    description="Single CSV file data asset",
    name="diabetes-local"
)
ml_client.data.create_or_update(my_data)
print("✓ URI_FILE data asset registered")

In [ ]:
# URI_FOLDER — points to a folder in a registered datastore
my_data = Data(
    path="azureml://datastores/blob_training_data/paths/data-asset-path/",
    type=AssetTypes.URI_FOLDER,
    description="Folder in blob datastore",
    name="diabetes-datastore-path"
)
ml_client.data.create_or_update(my_data)
print("✓ URI_FOLDER data asset registered")

In [ ]:
# MLTABLE — points to a folder containing an MLTable file
# The folder must contain a file named exactly 'MLTable' (no extension)
my_data = Data(
    path="/home/azureuser/cloudfiles/code/Users/Castle2696/azure-ml-labs/data",
    type=AssetTypes.MLTABLE,
    description="MLTable pointing to diabetes data folder",
    name="diabetes-table"
)
ml_client.data.create_or_update(my_data)
print("✓ MLTABLE data asset registered")

In [ ]:
# List all registered data assets
print("Registered data assets:")
for ds in ml_client.data.list():
    print(" -", ds.name)

In [ ]:
# Read an MLTable data asset into a Pandas DataFrame
# Requires: pip install mltable
import mltable

registered_data_asset = ml_client.data.get(name="diabetes-table", version=1)
tbl = mltable.load(f"azureml:/{registered_data_asset.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

In [ ]:
import mltable

latest = max(
    ml_client.data.list(name="diabetes-table"),
    key=lambda x: int(x.version)
)
print(f"Using version: {latest.version}")
tbl = mltable.load(f"azureml:/{latest.id}")
df = tbl.to_pandas_dataframe()
df.head(5)

---
## Section 4 — Compute

Esto tambien se puede hacer desde Cl

In [14]:
from azure.ai.ml.entities import AmlCompute

cpu_compute_target = "aml-cluster"

try:
    cpu_cluster = ml_client.compute.get(cpu_compute_target)
    print(f"✓ Compute '{cpu_compute_target}' already exists — reusing it")
except Exception:
    print(f"Creating compute cluster '{cpu_compute_target}'...")
    cpu_cluster = AmlCompute(
        name=cpu_compute_target,
        type="amlcompute",
        size="STANDARD_DS11_V2",
        min_instances=0,
        max_instances=2,
        idle_time_before_scale_down=120,
        tier="Dedicated",
    )
    cpu_cluster = ml_client.compute.begin_create_or_update(cpu_cluster)
    print(f"✓ Compute '{cpu_compute_target}' created")

Creating compute cluster 'aml-cluster'...


ClientAuthenticationError: (InvalidAuthenticationTokenTenant) The access token is from the wrong issuer 'https://sts.windows.net/f8cdef31-a31e-4b4a-93e4-5f571e91255a/'. It must match the tenant 'https://sts.windows.net/e43b0362-e932-4a58-9c0d-3071db3f47a8/' associated with this subscription. Please use the authority (URL) 'https://login.windows.net/e43b0362-e932-4a58-9c0d-3071db3f47a8' to get the token. Note, if the subscription is transferred to another tenant there is no impact to the services, but information about new tenant could take time to propagate (up to an hour). If you just transferred your subscription and see this error message, please try back later.
Code: InvalidAuthenticationTokenTenant
Message: The access token is from the wrong issuer 'https://sts.windows.net/f8cdef31-a31e-4b4a-93e4-5f571e91255a/'. It must match the tenant 'https://sts.windows.net/e43b0362-e932-4a58-9c0d-3071db3f47a8/' associated with this subscription. Please use the authority (URL) 'https://login.w

---
## Section 5.1 — Run a Training Job + Environment custom and curated

Important! dataset + script + conda-env.yml TOGETHER

In [ ]:
import os
os.makedirs("src", exist_ok=True)

In [ ]:
%%writefile src/diabetes-training.py
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print("Loading Data...")
diabetes = pd.read_csv('diabetes.csv')

X = diabetes[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure',
              'TricepsThickness','SerumInsulin','BMI','DiabetesPedigree','Age']].values
y = diabetes['Diabetic'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

reg = 0.01
print(f'Training LogisticRegression with regularization rate = {reg}')
model = LogisticRegression(C=1/reg, solver="liblinear").fit(X_train, y_train)

y_hat = model.predict(X_test)
acc = np.average(y_hat == y_test)
print('Accuracy:', acc)

y_scores = model.predict_proba(X_test)
auc = roc_auc_score(y_test, y_scores[:, 1])
print('AUC:', auc)

---
## Section 5.2 — Environments

In [ ]:
# List all environments (curated + custom)
print("Available environments:")
for env in ml_client.environments.list():
    print(" -", env.name)

Creating the custom environment. 

In [ ]:
from azure.ai.ml.entities import Environment

env_docker_image = Environment(
    image="mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04",
    name="docker-image-example",
    description="Environment created from a Docker image.",
)
ml_client.environments.create_or_update(env_docker_image)

First try to train the job with a custom environment

In [ ]:
from azure.ai.ml import command

# configure job
job = command(
    code="./src",
    command="python diabetes-training.py",
    environment="docker-image-example:1",
    compute="aml-cluster",
    display_name="diabetes-train-custom-env",
    experiment_name="diabetes-training"
)

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Custom an Environment

In [ ]:
%%writefile src/conda-env.yml
name: basic-env-cpu
channels:
  - conda-forge
dependencies:
  - python=3.11
  - scikit-learn
  - pandas
  - numpy
  - matplotlib

In [ ]:
from azure.ai.ml.entities import Environment

# Custom environment: base Docker image + conda dependencies
env_docker_conda = Environment(
    image="mcr.microsoft.com/azureml/openmpi3.1.2-ubuntu18.04",
    conda_file="./src/conda-env.yml",
    name="docker-image-plus-conda-example",
    description="Base Docker image with conda dependencies",
)
ml_client.environments.create_or_update(env_docker_conda)
print(f"✓ Environment '{env_docker_conda.name}' registered")

Running the job with new custom environment

In [ ]:
from azure.ai.ml import command

job = command(
    code="./src",
    command="python diabetes-training.py",
    environment="AzureML-sklearn-1.5@latest",  # Use curated env (safe default)
    compute="aml-cluster",
    display_name="diabetes-pythonv2-train",
    experiment_name="diabetes-training"
)

returned_job = ml_client.jobs.create_or_update(job)   # FIX: use ml_client.jobs, not ml_client directly
print("Monitor your job at:", returned_job.studio_url)

---
## Section 6 — MLflow Tracking

In [ ]:
import mlflow

# Point MLflow to your Azure ML workspace
mlflow_tracking_uri = ml_client.workspaces.get(ml_client.workspace_name).mlflow_tracking_uri
mlflow.set_tracking_uri(mlflow_tracking_uri)
print("✓ MLflow tracking URI set:", mlflow_tracking_uri)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Load data locally for notebook experimentation
diabetes = pd.read_csv('./diabetes.csv')
X = diabetes[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure',
              'TricepsThickness','SerumInsulin','BMI','DiabetesPedigree','Age']].values
y = diabetes['Diabetic'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

mlflow.set_experiment("diabetes-tracking")

with mlflow.start_run():
    model = DecisionTreeClassifier().fit(X_train, y_train)
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)

    # ROC curve artifact
    y_scores = model.predict_proba(X_test)
    fpr, tpr, _ = roc_curve(y_test, y_scores[:, 1])
    fig = plt.figure(figsize=(6, 4))
    plt.plot([0, 1], [0, 1], 'k--')
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("ROC-Curve.png")
    plt.show()

    mlflow.log_param("estimator", "DecisionTreeClassifier")
    mlflow.log_metric("Accuracy", acc)
    mlflow.log_artifact("ROC-Curve.png")

    print(f"✓ Run logged — Accuracy: {acc:.4f}")

---
## Section 8 — AutoML

In [ ]:
from azure.ai.ml import automl, Input
from azure.ai.ml.constants import AssetTypes

# Input must be an MLTable for AutoML
my_training_data_input = Input(
    type=AssetTypes.MLTABLE,
    path="azureml:diabetes-training:1"
)

classification_job = automl.classification(
    compute="aml-cluster",
    experiment_name="auto-ml-class-dev",
    training_data=my_training_data_input,
    target_column_name="Diabetic",
    primary_metric="accuracy",
    n_cross_validations=5,
    enable_model_explainability=True
)

classification_job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=5,
    enable_early_termination=True,
)

classification_job.set_training(
    blocked_training_algorithms=["LogisticRegression"],
    enable_onnx_compatible_models=True
)

# Submit
returned_job = ml_client.jobs.create_or_update(classification_job)
print("Monitor AutoML job at:", returned_job.studio_url)